In [ ]:
%load_ext autoreload
%autoreload 2

# Symb-xMIL on the synthetic MNIST toy task

Each bag is a set of MNIST digits. For the AND task a bag is positive iff it contains both a `4` and a `7`.
We train a MIL model, then explain it with Symb-xMIL over the semantic concepts
$\mathcal{N} = \{4, 7, \{0,1,2,3,5,6,8,9\}\}$. A model that learned the AND rule should be explained by `(4 ∧ 7)`.

In [ ]:
import os
import sys

import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

sys.path.insert(0, "..")               # repository root -> symb_xmil
sys.path.insert(0, "../models/xMIL")   # xMIL models and toy_experiments

from symb_xmil import QueryGenerator, run_symbolic_xmil
from toy_experiments.datasets import (
    AndDataset, OrDataset, MatDataset, FEATURE_PATHS, bag_collate_fn, get_MNIST_features,
)
from toy_experiments.models import get_model_and_classifier
from toy_experiments.scripts import train_model
from toy_experiments.symbolic import ToyPerturbationScorer, toy_concepts

## 1. Cache MNIST features (one-time)

Downloads MNIST, extracts ImageNet-pretrained ResNet18 features and stores them per digit as `class_{0..9}.pt`
in `FEATURE_PATHS["mnist_resnet18"]`. Set the `SYMB_XMIL_TOY_FEATURES` environment variable to use another root.
The cell is skipped if the cache already exists.

In [ ]:
mnist_root = "data/mnist"
feature_dir = FEATURE_PATHS["mnist_resnet18"]
print("feature cache:", feature_dir)

if not os.path.exists(os.path.join(feature_dir, "class_9.pt")):
    os.makedirs(feature_dir, exist_ok=True)
    features = get_MNIST_features(mnist_root, download=True)
    for digit, feats in features.items():
        torch.save(feats, os.path.join(feature_dir, f"class_{digit}.pt"))

## 2. Configuration

In [ ]:
dataset_cls = AndDataset # 4 AND 7 Dataset, you could also use OrDataset (4 OR 7), MatDataset (4 -> 7)
num_numbers = 10
num_bags_train = 2000
num_bags_val = 500
num_bags_test = 200
num_instances = 30
threshold = 1
noise = 0.0
sampling = "hierarchical"
features_type = "mnist_resnet18"  # or "onehot" (no feature cache needed)

model_type = "transmil"  # or "attention_mil"
model_dims = 20
dropout = False
n_out_layers = 0

batch_size = 1
num_epochs = 100
learning_rate = 0.001
weight_decay = 0.0001
patience = 5
tolerance = 0.0
warmup = 10

device = "cuda" if torch.cuda.is_available() else "cpu"
results_dir = "results/toy_and"

## 3. Train a MIL model

In [ ]:
dataset_train = dataset_cls(num_numbers, num_bags_train, num_instances, noise, threshold, sampling, features_type)
dataset_val = dataset_cls(num_numbers, num_bags_val, num_instances, noise, threshold, sampling, features_type)
collate_fn = bag_collate_fn if model_type == "attention_mil" else None
data_loader_train = DataLoader(dataset_train, batch_size=batch_size, collate_fn=collate_fn)
data_loader_val = DataLoader(dataset_val, batch_size=batch_size, collate_fn=collate_fn)

In [ ]:
model, classifier = get_model_and_classifier(
    model_type=model_type,
    num_features=dataset_train.num_features,
    num_classes=dataset_train.num_classes,
    model_dims=model_dims,
    dropout=dropout,
    n_out_layers=n_out_layers,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    device=device,
)

In [ ]:
train_metrics = train_model(
    classifier=classifier,
    num_classes=dataset_train.num_classes,
    data_loader_train=data_loader_train,
    data_loader_val=data_loader_val,
    batch_size=batch_size,
    num_epochs=num_epochs,
    warmup=warmup,
    tolerance=tolerance,
    patience=patience,
)

## 4. Symb-xMIL explanation

The test set uses `pure` sampling, so every bag contains a `4`, a `7` and at least one other digit.

1. For every bag $\mathbf{X}$ and every non-empty subset $\mathcal{S} \subseteq \mathcal{N}$, `ToyPerturbationScorer` builds the sub-bag
   $\mathbf{X}_\mathcal{S}$ of instances whose digit is in $\mathcal{S}$ and records the model output $f(\mathbf{X}_\mathcal{S})$
   (softmax probability of the explained class).
2. `QueryGenerator` builds Boolean queries $q$ over $\mathcal{N}$ (e.g. `4`, `(4 ∧ 7)`, `(4 ∨ 7)`) and their indicators $\lambda_q(\mathcal{S})$ over the same subsets.
3. The alignment score $r_\mathbf{X}(q, f)$ is the Pearson correlation between $f(\mathbf{X}_\mathcal{S})$ and $\lambda_q(\mathcal{S})$ across subsets;
   the best-aligned query $q^*(\mathbf{X})$ is the bag's symbolic explanation.

In [ ]:
dataset_test = dataset_cls(num_numbers, num_bags_test, num_instances, noise, threshold, "pure", features_type)
data_loader_test = DataLoader(dataset_test, batch_size=1, collate_fn=collate_fn)

# User-defined concepts N = [4, 7, (0, 1, 2, 3, 5, 6, 8, 9)]
concepts = toy_concepts(num_numbers)

# User-defined queries: k, negation and the semantic map M
query_gen = QueryGenerator(
    max_query_size=3,                # k
    semantic_map="toy_experiment",   # M: digit -> concept
    negation=False,
)
scorer = ToyPerturbationScorer(
    classifier=classifier,
    explained_class=-1,  # positive class
    device=torch.device(device),
)

results = run_symbolic_xmil(
    scorer=scorer,
    dataloader=data_loader_test,
    results_dir=results_dir,
    query_gen=query_gen,
    concepts=concepts,
    max_subset_size=None,  # all 2^|N| - 1 = 7 non-empty subsets S
    verbose=True,
)
f_df, best_df = results["f_df"], results["best_df"]

## 5. Results

`f_df`: $f(\mathbf{X}_\mathcal{S})$ per bag $\mathbf{X}$ (rows) and subset $\mathcal{S}$ (columns).
`best_df`: best-aligned query $q^*(\mathbf{X})$ per bag and its alignment score $r_\mathbf{X}(q, f)$ with every query $q$.

In [ ]:
f_df.head()

In [ ]:
best_counts = best_df["best_query"].value_counts()
print(best_counts)

ax = best_counts.plot.barh(figsize=(6, 0.4 * len(best_counts) + 1))
ax.invert_yaxis()
ax.set_xlabel("number of bags")
ax.set_title("Best symbolic query per bag")
plt.tight_layout()

In [ ]:
# Mean alignment score r_X(q, f) of each query across bags
best_df.drop(columns=["best_query", "best_alignment_score"]).select_dtypes("number").mean().sort_values(ascending=False).head(10)